# Pancancer enrichment analysis step 9: Try GSEApy PreRank

## Setup

In [1]:
import pandas as pd
import numpy as np
import os
import datetime
import gseapy as gp
import statsmodels.stats.multitest as sm

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
NUM_PERMUTATIONS = 1000

STEP01_DIR = "step01_outputs"
STEP01_FILE = "tumor_change_20200529_195104_10000_perm.tsv"
STEP01_FILE_PATH = os.path.join(STEP01_DIR, STEP01_FILE)

STEP09_DIR = "step09_outputs"

# Create log dir and file
LOG_DIR = "step09_checkpoints"
LOG_FILE = f"{TIME_START}_prerank.log"
LOG_FILE_PATH = os.path.join(LOG_DIR, LOG_FILE)

if not os.path.isdir(LOG_DIR):
    os.mkdir(LOG_DIR)
    
with open(LOG_FILE_PATH, 'w') as fp: 
    fp.write(f"{TIME_START}\n")
    fp.write(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Started step 9 with {NUM_PERMUTATIONS} permutations.\n")

GSEAPY_DIR_PATH = os.path.join(STEP09_DIR, "gseapy")

if not os.path.isdir(STEP09_DIR):
    os.mkdir(STEP09_DIR)

## Prepare data

In [3]:
# Read in the file from step 1
data = pd.read_csv(STEP01_FILE_PATH, sep='\t', index_col=0)
data = data[["cancer_type", "protein_str", "P_val"]]

In [4]:
# Replace zeros with 0.00001, then adjust the p values
data["P_val"] = data["P_val"].replace(to_replace=0, value=0.00001)

reject_null, adj_p, alpha_sidak, alpha_bonf = sm.multipletests(
    pvals=data["P_val"],
    alpha=0.05,
    method="fdr_bh")

data = data.\
    assign(adj_p=adj_p).\
    drop(columns="P_val")

data = data.\
    groupby(["cancer_type", "protein_str"]).\
    mean().\
    reset_index()

In [5]:
data

,cancer_type,protein_str,adj_p
0,ccrcc,A1BG,0.000020
1,ccrcc,A1CF,0.000020
2,ccrcc,A2M,0.000020
3,ccrcc,A4GALT,0.214310
4,ccrcc,AAAS,0.000020
...,...,...,...
84365,ovarian,ZWINT,0.852436
84366,ovarian,ZYG11B,0.258770
84367,ovarian,ZYX,0.000020
84368,ovarian,ZZEF1,0.000188


## Run GSEApy PreRank analysis

In [6]:
def gseapy_prerank_reactome(ranks, asc, wst):
    
    with open(LOG_FILE_PATH, 'a') as fp: 
        fp.write(f"\n{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Started asc={asc} wst={wst}.\n")
    
    output_file = f"enrichment_gseapy_prerank_{TIME_START}_{NUM_PERMUTATIONS}_perms_asc_{asc}_wst_{wst}.tsv"
    output_path = os.path.join(STEP09_DIR, output_file)
    
    # For each cancer, find enriched pathways.
    all_enrichments = pd.DataFrame()

    for cancer_type in ranks["cancer_type"].unique():

        prot = ranks.loc[
            ranks["cancer_type"] == cancer_type,
            ["protein_str", "adj_p"]
        ]
        
        pre_res = gp.prerank(
            rnk=prot,
            gene_sets="gene_set_libraries/Reactome_2020.gmt",
            outdir=GSEAPY_DIR_PATH,
            permutation_num=NUM_PERMUTATIONS,
            min_size=1,
            max_size=500, 
            weighted_score_type=wst,
            ascending=asc,
            no_plot=True,
            processes=1,
            seed=0)

        cancer_enriched = pre_res.res2d.assign(cancer_type=cancer_type)
        all_enrichments = all_enrichments.append(cancer_enriched)

        # Log that we finished this cancer type
        with open(LOG_FILE_PATH, 'a') as fp: 
            fp.write(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Finished {cancer_type} data.\n")
            
    # Record the results
    all_enrichments.to_csv(output_path, sep="\t")
            
    with open(LOG_FILE_PATH, 'a') as fp: 
        fp.write(f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - Finished asc={asc} wst={wst}.\n")

In [7]:
gseapy_prerank_reactome(data, asc=True, wst=1)